# x_true uncertainty in fit_trans: Morris GEX (200-gene subset)
Cis gene **HHEX**, GPU **cuda:6**. The model is rebuilt from the **production** trans config (`publication_runs/morris/slurm/configs/<label>_trans.yaml`), as `run_trans.py` does. After the production `min_log2_mu_ntc` exclusion, the trans panel is cut to 200 genes:
* every Domingo trans gene present in the panel, then
* the steepest **dependent** Hill components from the production full-transcriptome trans summary for this cis gene, alternating most-positive and most-negative `n` until 200 genes. Genes already in the Domingo set are not counted twice.

Only `fit_trans` is rerun, once per x-input condition.

In [ ]:
%load_ext autotime
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.abspath('.'))   # run from testing/xtrue_uncertainty/
import xunc_helpers as H
from config_utils import normalize_stage_args, is_enabled
print(os.environ.get('CONDA_DEFAULT_ENV'), torch.cuda.is_available())

In [ ]:
# Parameters
CIS_GENE     = 'HHEX'
LABEL_PREFIX = 'morris_20260806'
N_TRANS      = 200
# The morris_20260806 configs predate two library changes; rebuilding from them
# with current code would not match how the production fits were made:
#  - guide_covariates=[lane] was ignored in high-MOI mode when the cis fits ran
#    (48 unsplit guide effects); current code splits guides by lane.
#  - the production NTC/trans fits are keyed by gene_name with duplicates
#    suffixed -1, -2, ...; current code would use Ensembl IDs (and rejects
#    duplicate names in feature_name_col), so the NTC fit would match nothing.
MODEL_OVERRIDES = {'guide_covariates': [], 'guide_covariates_ntc': []}
LEGACY_FEATURE_NAMES_COL = 'gene_name'
DOMINGO_GENE_COUNTS = '/cfs/klemming/projects/snic/lappalainen_lab1/users/Leah/data/Domingo2024/processed_Leah/preprocessed/gene_counts.csv'
DEVICE           = 6   # GPU index on the node
TEST_OUTPUT_DIR  = '/cfs/klemming/projects/snic/lappalainen_lab1/users/Leah/CRISPRmodelling/BayesianModelling/xtrue_uncertainty'
X_DRAWS_PER_STEP = 1
REFIT            = False   # True: refit even if a test fit is already saved

CONDITIONS = H.CONDITIONS   # cell_median (old behaviour), cell_sample, guide_sample
device = f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu'

## Rebuild the production trans model

In [ ]:
PROD_LABEL = f'{LABEL_PREFIX}_{CIS_GENE}'
LABEL = f'xunc_{PROD_LABEL}'
model, cfg = H.build_trans_model(H.production_config('morris', PROD_LABEL), device,
                                 model_overrides=MODEL_OVERRIDES,
                                 legacy_feature_names_col=LEGACY_FEATURE_NAMES_COL)
n_guides, G_cis = model.guide_assignment.shape[1], model.posterior_samples_cis['x_eff_g'].shape[-1]
assert n_guides == G_cis, f'guide panel ({n_guides}) != cis fit guide effects ({G_cis})'
MODALITY = model.primary_modality

excl = cfg.get('exclude_trans_genes') or {}
if is_enabled(excl, default=False):
    model.exclude_trans_genes(**normalize_stage_args(excl))
print(model.get_modality(MODALITY))

## Select the 200 trans genes

In [ ]:
domingo_genes = pd.read_csv(DOMINGO_GENE_COUNTS, usecols=[0]).iloc[:, 0].astype(str).tolist()
summary_path = os.path.join(H.production_dir(cfg), f'trans_feature_summary_{MODALITY}.csv')

selection = H.select_morris_genes(model, summary_path, domingo_genes, n_total=N_TRANS, modality_name=MODALITY)
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
selection.to_csv(os.path.join(TEST_OUTPUT_DIR, f'{LABEL}_selected_genes.csv'), index=False)

keep = set(selection['feature'])
model.exclude_trans_genes(genes=[g for g in model.get_modality(MODALITY).feature_ids if g not in keep],
                          modality_name=MODALITY)
print(model.get_modality(MODALITY))
selection.value_counts('source')

## How noisy is x?
The per-cell posterior SD of log2 `x_true`, compared with the within-guide spread of the per-cell medians (the ordering the old fit treated as exact) and the spread between guides.

In [ ]:
x_unc = H.x_uncertainty_report(model)

## Fit trans under each x-input condition
Each condition is a deep copy of the prepared model, refit with `fit_trans(x_level=..., x_uncertainty=...)` and saved to `TEST_OUTPUT_DIR/<label>_<condition>/`. If that directory already has a saved fit, it is loaded instead (set `REFIT = True` to refit). Production outputs are never written.

In [ ]:
fit_kwargs = normalize_stage_args((cfg.get('trans') or {}).get('fit'))
fit_kwargs.pop('checkpoint_dir', None); fit_kwargs.pop('modality_name', None)
fit_kwargs['x_draws_per_step'] = X_DRAWS_PER_STEP
print(fit_kwargs)

models, summaries = H.fit_conditions(model, fit_kwargs, TEST_OUTPUT_DIR, LABEL,
                                     conditions=CONDITIONS, modality_name=MODALITY, refit=REFIT)

## Compare conditions
Counts of dependent genes, by active Hill component, then each gene's Hill coefficients against `cell_median` (the old behaviour).

In [ ]:
H.dependence_table(summaries)

In [ ]:
fig = H.plot_n_shift(summaries)
plt.show()

Did the steep genes stay steep? Production full-transcriptome n against each condition's n, for the steep-selected genes.

In [ ]:
steep = selection[selection['source'] != 'domingo'].set_index('feature')
fig, axes = plt.subplots(1, len(summaries), figsize=(4 * len(summaries), 3.6), squeeze=False)
for ax, (name, df) in zip(axes[0], summaries.items()):
    d = df.set_index('feature').reindex(steep.index)
    n_cond = np.where(steep['component'] == 'a', d['n_a_median'], d['n_b_median'])
    ax.scatter(steep['n'], n_cond, s=8, c=H.COND_COLORS.get(name, 'k'))
    lim = np.nanmax(np.abs(np.r_[steep['n'], n_cond])) * 1.05
    ax.plot([-lim, lim], [-lim, lim], ls='--', c='grey', lw=0.8)
    ax.set_xlabel('production n (full panel)'); ax.set_ylabel(f'n ({name})'); ax.set_title(name)
fig.tight_layout(); plt.show()

## Trans fits, all genes
One panel per feature, with every condition's fitted curve and 95% CI overlaid. Panel titles give `which_active` per condition (cm = cell_median, cs = cell_sample, gs = guide_sample). All pages also go to `overlays.pdf`.

In [ ]:
features = sorted(summaries['cell_median']['feature'])
H.plot_condition_overlays(models, summaries, features, modality_name=MODALITY,
                          pdf_path=os.path.join(TEST_OUTPUT_DIR, f'{LABEL}_overlays.pdf'),
                          use_log2fc=True)

## Data + fit per condition (saved, not shown)
`plot_xy_data` for every gene × condition → `TEST_OUTPUT_DIR/<label>_xy/<condition>/<gene>.png`. In every condition the x positions are the per-cell posterior-median `x_true`, including `guide_sample`, whose fit actually used guide-level x. Pass `features=[...]` to `plot_xy_data` directly to look at a few inline.

In [ ]:
H.save_xy_plots(models, features, os.path.join(TEST_OUTPUT_DIR, f'{LABEL}_xy'),
                modality_name=MODALITY, sum_factor_col=fit_kwargs['sum_factor_col'], color_by='targeting',
                color_palette={'NTC': 'gray', 'Targeting': 'steelblue'},
                log2fc=True, show_correction='corrected', legend_outside=True, figsize=(5, 3))